# EDA 02: Source × district coverage audit  *(TODO: Sage)*

**EDA step 2.** For every source in the plan, which of the 75 districts does it cover, for
which variables and which years, and is it registered with a licence? The answer decides
*keep / drop / proxy* per variable (step 4) and feeds the confidence rating.

| | |
|---|---|
| **Inputs** | the manifests `data/raw/*/_manifest.jsonl` (what was downloaded, from where, under which licence); the **tidy, LGD-keyed tables** the loader notebooks write to `data/interim/<topic>/` (EDA 10 `apy`, EDA 11 `climate`, then one per manual download); the spine |
| **Outputs** | `data/interim/coverage/coverage_long.parquet`, `.../coverage_grid.parquet`, `docs/eda/02_coverage.md`, new rows in `docs/eda/data-quality-log.md` |
| **Gate** | "source × district coverage grid" (Phase 1, step 2) |

What changed in session 2: the loader notebooks already map every source to LGD codes with
`map_to_lgd`, so this notebook no longer loads raw files. It reads the manifests (section 1)
and the tidy tables (section 2). Sources still waiting for a manual download show up as
gaps, which is the point. Same pattern as before: fill the empty cells (comments are the
pseudocode); each section ends with an `assert` that must pass.

In [ ]:
# The "noqa: F401" marks: these imports are used once you fill in the TODO cells; remove the noqa then.
import duckdb  # noqa: F401
import pandas as pd

from up_siting.common.paths import DOCS_DIR, INTERIM_DIR, RAW_DIR, SEEDS_DIR  # noqa: F401

spine = pd.read_csv(
    SEEDS_DIR / "lgd_district_spine.csv", dtype={"census_2001_code": str, "census_2011_code": str}
)
lineage = pd.read_csv(SEEDS_DIR / "district_lineage.csv")
OUT_DIR = INTERIM_DIR / "coverage"

# The Phase 1 source list, ethanol first (docs/eda/downloads.md has the cards).
PLANNED = {
    # source_id: (pilot, priority)
    "apy_idp": ("ethanol", 0), "wikidata_hq": ("ethanol", 0), "nasa_power": ("ethanol", 0),
    "cgwb_gw": ("ethanol", 1), "sansad_cane": ("ethanol", 2), "agmarknet_ceda": ("ethanol", 3),
    "dfpd_distilleries": ("ethanol", 4), "industrial_land": ("ethanol", 5),
    "igrs_circle_rates": ("ethanol", 6), "sugar_mills": ("ethanol", 7),
    "niti_ethanol_roadmap": ("ethanol", 7), "uperc_tariff": ("ethanol", 11),
    "sansad_odop": ("ethanol", 12), "apy_des": ("ethanol", 13), "icrisat_dld": ("ethanol", 14),
    "livestock_census": ("cbg", 8), "residue_ratios": ("cbg", 9), "cbg_plants": ("cbg", 10),
}  # fmt: skip
print(len(spine), "districts;", len(PLANNED), "planned sources")

## 1. Which sources are downloaded and registered, and under which licence?

Every file in `data/raw/` has a manifest line (`make register`). Read them all at once
with DuckDB and compare with `PLANNED`.

In [ ]:
# TODO(sage)
# 1. manifests = all lines of data/raw/*/_manifest.jsonl as one DataFrame.
#    DuckDB: read_json('<RAW_DIR>/*/_manifest.jsonl'). Columns: file, source, ingest_date,
#    sha256, bytes, source_url, fetched_at, fetched_by, licence, notes.
# 2. registry = one row per PLANNED source_id: pilot, priority, n_files, total_mb,
#    latest_ingest_date, fetched_by (manual/extractor), licence (first one),
#    status = "registered" if it has files else "waiting for download".
#    Hint: build a DataFrame from PLANNED, left join the per-source aggregate.
# 3. licence_flag = True where the licence starts with "⚠" or contains "Unknown"/"Not stated".
manifests: pd.DataFrame | None = None
registry: pd.DataFrame | None = None

In [ ]:
assert isinstance(manifests, pd.DataFrame) and isinstance(registry, pd.DataFrame)
assert set(registry["source_id"]) == set(PLANNED)
assert manifests["sha256"].is_unique, "the same file registered twice under different names?"
assert (manifests["source"].isin(PLANNED) | manifests["source"].eq("lgd")).all()
# Ethanol sources loaded in session 2 must be there:
assert (
    registry.set_index("source_id")
    .loc[["apy_idp", "nasa_power", "wikidata_hq"], "status"]
    .eq("registered")
    .all()
)

## 2. Which tidy tables exist, and what does each measure?

Register each **tidy table** once (not each raw file). One entry per *variable*: a small
DuckDB query that returns `lgd_district_code`, `year` and `value` from the loader's
Parquet. Two entries are filled in as examples; add one per variable as the loaders for the
manual downloads arrive.

In [ ]:
APY = (INTERIM_DIR / "apy" / "apy_up_long.parquet").as_posix()
POWER = (INTERIM_DIR / "climate" / "power_daily.parquet").as_posix()

VARIABLES = [
    {
        "source_id": "apy_idp",
        "variable": "sugarcane_production_t",
        "sql": f"""
            select lgd_district_code, crop_year as year, sum(production_t) as value
            from '{APY}'
            where crop = 'sugarcane' and (resource = 'upag' or crop_year < 2013)
            group by all""",
    },
    {
        "source_id": "nasa_power",
        "variable": "ghi_kwh_m2_day",
        "sql": f"""
            select lgd_district_code, year(date) as year, avg(ghi_kwh_m2_day) as value
            from '{POWER}' group by all""",
    },
    # TODO(sage): maize_production_t, rice_production_t, wheat_production_t (same as cane),
    # precip_mm_per_year, hot_days_ge_40c, then cgwb_gw, agmarknet_ceda, ... as they load.
]

In [ ]:
# TODO(sage)
# 1. For each entry: frame = duckdb.sql(entry["sql"]).df(); add source_id and variable.
# 2. Concatenate into `values_long` (source_id, variable, year, lgd_district_code, value).
# 3. Check every lgd_district_code is in the spine (the loaders guarantee it; trust but verify).
values_long: pd.DataFrame | None = None

In [ ]:
assert isinstance(values_long, pd.DataFrame)
assert {"source_id", "variable", "year", "lgd_district_code", "value"} <= set(values_long.columns)
assert values_long["lgd_district_code"].isin(spine["lgd_district_code"]).all()
assert not values_long.duplicated(["variable", "year", "lgd_district_code"]).any()

## 3. Build the coverage grid: for each source and district, how much data is there?

Two tables. The **long** one is the real result; the **grid** is just a pivot of it for
reading.

* `coverage_long`: one row per (source_id, variable, year, lgd_district_code) with
  `n_rows` and `share_non_null` (0–1), built from `values_long`. **All 75 districts must appear for every
  source/variable**, including the ones with no data (`share_non_null = 0`), or you can't
  see the gaps. Hint: cross join the spine with the source/variable list, then left join
  the counts.
* `coverage_grid`: index = lgd_district_code (75 rows), one column per `source_id.variable`,
  values = share_non_null.

In [ ]:
# TODO(sage)
# 1. From values_long: group by source_id, variable, year, lgd_district_code; count rows
#    and non-null values.
# 2. Build every (source_id, variable, year) x 75 district combination and left join step 1,
#    filling missing counts with 0. DuckDB SQL over the DataFrames is a good fit here.
# 3. share_non_null = n_non_null / n_rows (0 where n_rows is 0).
# 4. Pivot into coverage_grid.
coverage_long: pd.DataFrame | None = None
coverage_grid: pd.DataFrame | None = None

In [ ]:
assert isinstance(coverage_long, pd.DataFrame) and isinstance(coverage_grid, pd.DataFrame)
expected_cols = {"source_id", "variable", "year", "lgd_district_code", "n_rows", "share_non_null"}
assert expected_cols <= set(coverage_long.columns)
per_combo = coverage_long.groupby(["source_id", "variable", "year"])["lgd_district_code"].nunique()
assert (per_combo == 75).all(), per_combo[per_combo != 75]
assert coverage_long["share_non_null"].between(0, 1).all()
assert len(coverage_grid) == 75 and set(coverage_grid.index) == set(spine["lgd_district_code"])

## 4. Where are the gaps, and are they explained?

Summaries a reviewer can read in ten seconds:

* `by_source`: per source/variable, the number of districts with `share_non_null > 0`.
* `by_district`: per district, how many source/variables have no data, plus `in_census_2011`
  from the spine, because the four post-2011 districts are the expected suspects.

Then look: is a gap explained by a district split (lineage table)? A source that only
covers some regions? A naming problem you missed?

In [ ]:
# TODO(sage)
# 1. by_source: group coverage_long by source_id, variable; count districts with share > 0.
# 2. by_district: count, per district, the source/variables with share == 0; join spine
#    (district_name, in_census_2011); sort by the count, worst first.
# 3. Optional: a heatmap of coverage_grid (districts x sources) with matplotlib's imshow.
by_source: pd.DataFrame | None = None
by_district: pd.DataFrame | None = None

In [ ]:
assert isinstance(by_source, pd.DataFrame) and isinstance(by_district, pd.DataFrame)
assert len(by_district) == 75
assert {"district_name", "in_census_2011"} <= set(by_district.columns)

## 5. Write outputs and record findings

Write `coverage_long` and `coverage_grid` to `OUT_DIR` as Parquet and read them back (the
file is the deliverable). Then fill in the findings below, copy them to
`docs/eda/02_coverage.md`, and add a data-quality log row for every gap or surprise.

In [ ]:
# TODO(sage): mkdir OUT_DIR; write both frames to Parquet; read back with DuckDB and compare row counts.

In [ ]:
assert (OUT_DIR / "coverage_long.parquet").is_file()
assert (OUT_DIR / "coverage_grid.parquet").is_file()

## 6. Findings  *(TODO: Sage)*

1. *Sources registered, and which ones cover all 75 districts …*
2. *Gaps and why (split districts, partial coverage, missing years) …*
3. *Sources still waiting for a download, and what that blocks …*
4. *Licence issues spotted while downloading …*